# XRDBench dataset explorer

This notebook previews the three XRDBench datasets: **MP500**, **RRUFF**, and **opXRD**. For every dataset it displays two single-phase and two multi-phase examples. Each example includes its XRD pattern and reference crystal structure(s).

> Dataset archives must be downloaded from GitHub Releases and extracted under `datasets/` before running this notebook.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
from ase.io import read
from ase.visualize.plot import plot_atoms
from IPython.display import Markdown, display

DATASETS = ("mp500", "rruff", "opxrd")
SAMPLES_PER_MODE = 2
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "datasets"

missing = [name for name in DATASETS if not (DATA_ROOT / name / "patterns.h5").is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing dataset archives: {', '.join(missing)}. Download them from \
        https://github.com/Asterbin/xrdbench/releases and extract them under datasets/."
    )

print(f"Dataset root: {DATA_ROOT.resolve()}")

In [ ]:
def load_manifest(dataset_root: Path) -> list[dict]:
    with (dataset_root / "manifest.jsonl").open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def get_pattern(dataset_root: Path, record: dict):
    location = record["input"]
    with h5py.File(dataset_root / location["archive"], "r") as h5:
        return h5["two_theta_deg"][:], h5[location["dataset"]][location["index"]]


def get_structure_paths(dataset_root: Path, record: dict) -> list[Path]:
    return [dataset_root / path for path in record["ground_truth"]["structure_files"]]


def show_sample(dataset_root: Path, record: dict) -> None:
    two_theta, intensity = get_pattern(dataset_root, record)
    structure_paths = get_structure_paths(dataset_root, record)
    structures = [read(path) for path in structure_paths]

    figure, axes = plt.subplots(1, 1 + len(structures), figsize=(6 + 3.5 * len(structures), 3.8))
    axes = [axes] if len(structures) == 0 else axes
    axes[0].plot(two_theta, intensity, color="#2563eb", linewidth=1)
    axes[0].set(title=record["sample_id"], xlabel=r"2θ (degrees)", ylabel="normalized intensity")
    axes[0].grid(alpha=0.2)

    phase_ids = record["ground_truth"]["phase_ids"]
    for index, (axis, atoms, phase_id) in enumerate(zip(axes[1:], structures, phase_ids), 1):
        plot_atoms(atoms, ax=axis, rotation=("35x,25y,10z"), radii=0.55, show_unit_cell=2)
        axis.set(title=f"phase {index}: {phase_id}")
        axis.axis("off")

    figure.tight_layout()
    display(Markdown(
        f"**Task:** `{record['task']}`  \\n"
        f"**Reference phase IDs (audit metadata):** `{', '.join(phase_ids)}`  \\n"
        f"**Reference CIF files:** `{', '.join(path.name for path in structure_paths)}`"
    ))
    plt.show()


In [ ]:
for dataset_name in DATASETS:
    dataset_root = DATA_ROOT / dataset_name
    manifest = load_manifest(dataset_root)
    single = [item for item in manifest if item["task"] == "identification.single"]
    multi = [item for item in manifest if item["task"] == "identification.multi"]

    display(Markdown(f"# {dataset_name.upper()}"))
    display(Markdown(f"Single-phase records: **{len(single):,}** · Multi-phase records: **{len(multi):,}**"))

    display(Markdown("## Single-phase examples"))
    for record in single[:SAMPLES_PER_MODE]:
        show_sample(dataset_root, record)

    display(Markdown("## Multi-phase examples"))
    for record in multi[:SAMPLES_PER_MODE]:
        show_sample(dataset_root, record)


## Next steps

- Change `SAMPLES_PER_MODE` in the first code cell to display more or fewer examples.
- The displayed phase IDs are internal audit metadata. A model should normally submit CIF files through `prediction.structure_files`, not these IDs.
- See `domo/README.zh-CN.md` for submission.jsonl examples and `docs/GETTING_STARTED.zh-CN.html` for the full user guide.